# Trace-Based Imitation

**Last updated:** 2026-04-09 12:00 PT

**Track:** Learning | **Sub-Ability:** Observational learning

Can the model learn a novel procedure by observing worked examples?
Pre/post learning framework: observe execution traces, then apply to new inputs.

**Difficulty grid:** procedure complexity x num traces x 3 seeds = 27 items

In [ ]:
import kaggle_benchmarks as kbench
import pandas as pd
import random
import re
import sqlite3
import json
import time
import urllib.request
import urllib.error

print('Available models:', list(kbench.llms.keys()))
print('Default model:', kbench.llm)

In [ ]:
def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines: return ''
    for line in reversed(lines):
        clean = line.strip('"\'\'\u201c\u201d').rstrip('.!?')
        if len(clean) <= 30:
            return clean.lower().strip()
    return lines[-1].strip('"\'\'\u201c\u201d').rstrip('.!?').lower().strip()

def extract_number(response: str) -> str:
    """Extract a number from model response."""
    text = strip_thinking(response)
    text = re.sub(r'[`*_]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    for line in reversed(lines):
        clean = line.strip('.!?, ')
        if re.match(r'^-?\d+$', clean): return clean
    nums = re.findall(r'-?\d+', text)
    return nums[-1] if nums else ''

In [ ]:
# === Novel procedure definitions ===
# These are invented algorithms that do NOT correspond to any standard named algorithm.
# Each takes a list of integers and returns a single integer via deterministic steps.

# --- SIMPLE (1-step) procedures ---

def zigzag_sum(nums):
    """Add odd-indexed elements, subtract even-indexed elements, multiply by count."""
    # Step 1: odd-indexed += , even-indexed -=, then * len
    total = 0
    for i, v in enumerate(nums):
        if i % 2 == 1:
            total += v
        else:
            total -= v
    result = total * len(nums)
    return result, [
        f"For each element: subtract if even-indexed, add if odd-indexed -> running total = {total}",
        f"Multiply by count of elements ({len(nums)}) -> {total} * {len(nums)} = {result}",
    ]

def mirror_diff(nums):
    """Pair first with last, second with second-to-last, etc. Sum absolute differences, add middle if odd length."""
    pairs = []
    total = 0
    n = len(nums)
    for i in range(n // 2):
        d = abs(nums[i] - nums[n - 1 - i])
        pairs.append(f"|{nums[i]} - {nums[n-1-i]}| = {d}")
        total += d
    middle_note = ""
    if n % 2 == 1:
        mid = nums[n // 2]
        total += mid
        middle_note = f", add middle element {mid}"
    result = total
    return result, [
        f"Pair from outside in and take absolute differences: {'; '.join(pairs)}",
        f"Sum of differences{middle_note} = {result}",
    ]

def triangle_count(nums):
    """Square each element, sum digits of each square, then take alternating sum of those digit-sums."""
    squares = [v * v for v in nums]
    digit_sums = [sum(int(d) for d in str(s)) for s in squares]
    alt_sum = 0
    for i, ds in enumerate(digit_sums):
        if i % 2 == 0:
            alt_sum += ds
        else:
            alt_sum -= ds
    return alt_sum, [
        f"Square each element: {nums} -> {squares}",
        f"Sum digits of each square: {squares} -> {digit_sums}",
        f"Alternating sum (+/-) of digit-sums: {' + '.join(str(ds) if i%2==0 else f'(- {ds})' for i,ds in enumerate(digit_sums))} = {alt_sum}",
    ]

# --- MEDIUM (2-step) procedures ---

def spiral_crunch(nums):
    """Step 1: Reverse every other pair. Step 2: Cumulative sum, return last."""
    # Step 1: reverse every other pair (pairs at index 0,1 / 2,3 / 4,5 ...)
    transformed = list(nums)
    for i in range(0, len(transformed) - 1, 4):  # swap pairs at positions 0-1, 4-5, 8-9, ...
        transformed[i], transformed[i + 1] = transformed[i + 1], transformed[i]
    # Step 2: cumulative sum
    cumsum = []
    running = 0
    for v in transformed:
        running += v
        cumsum.append(running)
    result = cumsum[-1] if cumsum else 0
    return result, [
        f"Swap every other pair (positions 0-1, 4-5, ...): {list(nums)} -> {transformed}",
        f"Cumulative sum: {transformed} -> {cumsum}",
        f"Final value = {result}",
    ]

def wave_hash(nums):
    """Step 1: Sort, interleave min/max (smallest, largest, 2nd smallest, 2nd largest...).
       Step 2: Multiply adjacent pairs, sum products."""
    s = sorted(nums)
    interleaved = []
    lo, hi = 0, len(s) - 1
    while lo <= hi:
        interleaved.append(s[lo])
        if lo != hi:
            interleaved.append(s[hi])
        lo += 1
        hi -= 1
    products = []
    for i in range(0, len(interleaved) - 1, 2):
        products.append(interleaved[i] * interleaved[i + 1])
    result = sum(products)
    return result, [
        f"Sort and interleave min/max: {list(nums)} -> sorted {s} -> interleaved {interleaved}",
        f"Multiply adjacent pairs: {[f'{interleaved[i]}*{interleaved[i+1]}' for i in range(0, len(interleaved)-1, 2)]} = {products}",
        f"Sum of products = {result}",
    ]

def cascade_mod(nums):
    """Step 1: Replace each element with (element * its 1-based position) mod 17.
       Step 2: XOR all results together, then multiply by count."""
    mod_vals = [(v * (i + 1)) % 17 for i, v in enumerate(nums)]
    xor_result = 0
    for v in mod_vals:
        xor_result ^= v
    result = xor_result * len(nums)
    return result, [
        f"Multiply each by its 1-based position, mod 17: {[f'{v}*{i+1}%17={m}' for i,(v,m) in enumerate(zip(nums, mod_vals))]}",
        f"XOR all results: {' ^ '.join(str(v) for v in mod_vals)} = {xor_result}",
        f"Multiply by count ({len(nums)}): {xor_result} * {len(nums)} = {result}",
    ]

# --- COMPLEX (3-step with conditionals) procedures ---

def branch_fold(nums):
    """If first element > 5: fold left with multiply (capped at 999), else fold right with add.
       Then negate odd-positioned results in the intermediate list.
       Finally, sum all."""
    if nums[0] > 5:
        branch = "left-multiply"
        acc = nums[0]
        intermediates = [acc]
        for v in nums[1:]:
            acc = (acc * v) % 999  # keep manageable
            intermediates.append(acc)
    else:
        branch = "right-add"
        acc = nums[-1]
        intermediates = [0] * len(nums)
        intermediates[-1] = acc
        for i in range(len(nums) - 2, -1, -1):
            acc = acc + nums[i]
            intermediates[i] = acc
    # Negate odd positions
    negated = [(-v if i % 2 == 1 else v) for i, v in enumerate(intermediates)]
    result = sum(negated)
    return result, [
        f"First element = {nums[0]}, {'> 5' if nums[0] > 5 else '<= 5'} -> branch: {branch}",
        f"Fold to get intermediates: {intermediates}",
        f"Negate odd-indexed positions: {intermediates} -> {negated}",
        f"Sum all: {' + '.join(str(v) for v in negated)} = {result}",
    ]

def pivot_weave(nums):
    """Step 1: Find median (middle after sort). Split into above/below median.
       Step 2: Interleave below and above lists, padding shorter with 0.
       Step 3: If length is even, sum all; if odd, alternating sum."""
    s = sorted(nums)
    median = s[len(s) // 2]
    below = [x for x in nums if x < median]
    above = [x for x in nums if x > median]
    at_med = [x for x in nums if x == median]
    # Put median values in below
    below = below + at_med
    # Interleave
    woven = []
    for i in range(max(len(below), len(above))):
        if i < len(below):
            woven.append(below[i])
        else:
            woven.append(0)
        if i < len(above):
            woven.append(above[i])
        else:
            woven.append(0)
    # Conditional final step
    if len(nums) % 2 == 0:
        result = sum(woven)
        step3 = f"Length {len(nums)} is even -> plain sum of woven: {result}"
    else:
        result = sum(v if i % 2 == 0 else -v for i, v in enumerate(woven))
        step3 = f"Length {len(nums)} is odd -> alternating sum of woven: {result}"
    return result, [
        f"Sort {list(nums)} -> {s}, median = {median}. Below/at median: {below}, above: {above}",
        f"Interleave (pad with 0): {woven}",
        step3,
    ]

def chain_gate(nums):
    """Step 1: Compute pairwise sums of consecutive elements.
       Step 2: For each pairwise sum, if > 10 keep it, else replace with 1.
       Step 3: Product of all gated values, mod 1000, then subtract first original element."""
    pair_sums = [nums[i] + nums[i + 1] for i in range(len(nums) - 1)]
    gated = [v if v > 10 else 1 for v in pair_sums]
    product = 1
    for v in gated:
        product *= v
    product_mod = product % 1000
    result = product_mod - nums[0]
    return result, [
        f"Pairwise sums of consecutive elements: {[f'{nums[i]}+{nums[i+1]}={pair_sums[i]}' for i in range(len(pair_sums))]}",
        f"Gate: keep if >10, else replace with 1: {pair_sums} -> {gated}",
        f"Product mod 1000: {'*'.join(str(v) for v in gated)} = {product} mod 1000 = {product_mod}",
        f"Subtract first original element ({nums[0]}): {product_mod} - {nums[0]} = {result}",
    ]

# === Procedure registry ===
PROCEDURES = {
    'simple': [
        ('zigzag_sum', zigzag_sum, "Zigzag Sum: subtract even-indexed, add odd-indexed, multiply by count"),
        ('mirror_diff', mirror_diff, "Mirror Diff: pair outside-in, sum absolute differences, add middle if odd"),
        ('triangle_count', triangle_count, "Triangle Count: square each, digit-sum each square, alternating sum"),
    ],
    'medium': [
        ('spiral_crunch', spiral_crunch, "Spiral Crunch: swap every other pair, then cumulative sum"),
        ('wave_hash', wave_hash, "Wave Hash: sort-interleave min/max, multiply adjacent pairs, sum"),
        ('cascade_mod', cascade_mod, "Cascade Mod: multiply by position mod 17, XOR all, multiply by count"),
    ],
    'complex': [
        ('branch_fold', branch_fold, "Branch Fold: conditional fold direction, negate odd positions, sum"),
        ('pivot_weave', pivot_weave, "Pivot Weave: split by median, interleave, conditional sum"),
        ('chain_gate', chain_gate, "Chain Gate: pairwise sums, gate >10, product mod 1000, subtract first"),
    ],
}

print("Procedure registry:")
for level, procs in PROCEDURES.items():
    for name, fn, desc in procs:
        # Quick smoke test
        test = [3, 7, 2, 8, 5]
        result, steps = fn(test)
        print(f"  [{level}] {name}: {test} -> {result}")
print("All procedures verified.")

In [ ]:
# === Dataset: complexity x traces x seeds ===
COMPLEXITY_LEVELS = ['simple', 'medium', 'complex']
TRACE_COUNTS = [2, 3, 4]  # few, mid, many worked examples
SEEDS = 3
rows = []
tid = 0

def generate_input(rng, length=5):
    """Generate a random list of integers for procedure input."""
    return [rng.randint(1, 9) for _ in range(length)]

def format_trace(inputs, fn):
    """Generate a full worked trace: input -> step1 -> step2 -> ... -> output."""
    result, steps = fn(inputs)
    trace = f"Input: {inputs}\n"
    for i, step in enumerate(steps, 1):
        trace += f"  Step {i}: {step}\n"
    trace += f"Output: {result}"
    return trace, result

for complexity in COMPLEXITY_LEVELS:
    procs = PROCEDURES[complexity]
    for n_traces in TRACE_COUNTS:
        for seed in range(SEEDS):
            rng = random.Random(seed * 1000 + TRACE_COUNTS.index(n_traces) * 100 + COMPLEXITY_LEVELS.index(complexity) * 10)
            # Pick a procedure for this cell (rotate through the 3 per level)
            proc_name, proc_fn, proc_desc = procs[seed % len(procs)]

            # Generate worked example traces
            traces = []
            for t in range(n_traces):
                ex_input = generate_input(rng)
                trace_str, _ = format_trace(ex_input, proc_fn)
                traces.append(trace_str)

            # Generate test input
            test_input = generate_input(rng)
            expected_result, expected_steps = proc_fn(test_input)

            material = "\n\n".join(f"Example {i+1}:\n{tr}" for i, tr in enumerate(traces))
            label = f"{complexity}_{n_traces}traces"

            rows.append({
                'task_id': tid,
                'seed': seed,
                'complexity': complexity,
                'n_traces': n_traces,
                'difficulty_label': label,
                'procedure_name': proc_name,
                'procedure_desc': proc_desc,
                'material': material,
                'test_input': str(test_input),
                'expected': str(expected_result),
                'expected_steps': '\n'.join(expected_steps),
                'item_desc': f'{proc_name} ({complexity}, {n_traces} traces, seed {seed})',
            })
            tid += 1

dataset = pd.DataFrame(rows)
print(f'Dataset: {len(dataset)} items')
print(f'Complexity levels: {dataset["complexity"].unique().tolist()}')
print(f'Trace counts: {dataset["n_traces"].unique().tolist()}')
print()
print(dataset[['task_id', 'difficulty_label', 'procedure_name', 'test_input', 'expected']].to_string(index=False))

In [ ]:
DB_PATH = 'trace_imitation_runs.db'
db = sqlite3.connect(DB_PATH)
db.execute('DROP TABLE IF EXISTS runs')
db.execute('''
    CREATE TABLE runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, model TEXT, task_id TEXT,
        complexity TEXT, n_traces INTEGER, difficulty_label TEXT,
        seed INTEGER, procedure_name TEXT, item_desc TEXT,
        test_input TEXT, expected TEXT,
        pre_raw TEXT, pre_extracted TEXT, pre_correct INTEGER,
        study_notes TEXT,
        post_raw TEXT, post_extracted TEXT, post_correct INTEGER,
        pre_time_s REAL, study_time_s REAL, post_time_s REAL
    )
''')
db.commit()
print(f'SQLite ready (fresh): {DB_PATH}')

In [ ]:
# === Prompt templates ===

PRE_P = """You are shown worked examples of a novel procedure applied to lists of numbers.
Each example shows the input, intermediate steps, and final output.

{material}

Apply the same procedure to this input: {test_input}

Show your work step by step, then give the final answer as a single number on the last line."""

STUDY_P = """You are shown worked examples of a novel procedure applied to lists of numbers.
Each example shows the input, intermediate steps, and final output.

{material}

Your task:
1. Analyze these examples step by step.
2. Write down the EXACT procedure being followed.
3. Be precise about the order of operations, any conditions or branching rules, and how each step transforms the data.
4. Verify your understanding by mentally re-running the procedure on at least one example.

Write clear, detailed notes that would let someone else reproduce this procedure exactly."""

POST_P = """You previously studied a procedure from worked examples and wrote these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Here are the original worked examples for reference:
{material}

Apply the procedure to this new input: {test_input}

Show your work step by step, then give the final answer as a single number on the last line."""

## Evaluation

In [ ]:
@kbench.task(name='trace_based_imitation',
             description='Pre/post learning: imitate novel procedures from worked traces')
def trace_based_imitation(llm, material: str, test_input: str, expected: str,
                          complexity: str, n_traces: int, difficulty_label: str,
                          seed: int, procedure_name: str, item_desc: str,
                          procedure_desc: str, expected_steps: str) -> dict:
    model_name = str(llm)
    ts = time.strftime('%Y-%m-%d %H:%M:%S')
    task_id = f'{complexity}_{n_traces}traces_{seed}'

    # PRE: apply procedure without study
    t0 = time.time()
    pre_raw = llm.prompt(PRE_P.format(material=material, test_input=test_input))
    pre_time = time.time() - t0
    pre_extracted = extract_number(pre_raw)
    pre_correct = pre_extracted == str(expected)

    # STUDY: analyze the traces and write notes
    t0 = time.time()
    study_raw = llm.prompt(STUDY_P.format(material=material))
    study_time = time.time() - t0
    notes = strip_thinking(study_raw)

    # POST: apply procedure with notes
    t0 = time.time()
    post_raw = llm.prompt(POST_P.format(notes=notes, material=material, test_input=test_input))
    post_time = time.time() - t0
    post_extracted = extract_number(post_raw)
    post_correct = post_extracted == str(expected)

    try:
        db.execute(
            """INSERT INTO runs (timestamp,model,task_id,complexity,n_traces,difficulty_label,
               seed,procedure_name,item_desc,test_input,expected,
               pre_raw,pre_extracted,pre_correct,
               study_notes,post_raw,post_extracted,post_correct,
               pre_time_s,study_time_s,post_time_s)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (ts, model_name, task_id, complexity, int(n_traces), difficulty_label,
             int(seed), procedure_name, item_desc, test_input, str(expected),
             pre_raw, pre_extracted, int(pre_correct),
             notes, post_raw, post_extracted, int(post_correct),
             pre_time, study_time, post_time))
        db.commit()
    except Exception as e:
        print(f'  [SQLite warn] {e}')

    b = 'Y' if pre_correct else 'N'
    l = 'Y' if post_correct else 'N'
    print(f'[{difficulty_label:18s}] {procedure_name:15s} expected={expected:>6s}  '
          f'pre="{pre_extracted}"({b})  post="{post_extracted}"({l})  '
          f'times: {pre_time:.1f}s/{study_time:.1f}s/{post_time:.1f}s')
    return {'pre_correct': pre_correct, 'post_correct': post_correct}

runs = trace_based_imitation.evaluate(llm=[kbench.llm], evaluation_data=dataset.set_index('task_id'),
                                      n_jobs=1, timeout=3600, max_attempts=1)
print(f'\nDone: {len(runs.as_dataframe())} items')

## Results

In [ ]:
results = pd.read_sql('SELECT * FROM runs ORDER BY task_id', db)
print(f'Total runs: {len(results)}\n')

pre_acc = results['pre_correct'].mean()
post_acc = results['post_correct'].mean()
gain = post_acc - pre_acc
room = 1.0 - pre_acc
eff = gain / room if room > 0 else 0.0

print('=' * 60)
print(f'Model: {results["model"].iloc[0] if len(results) > 0 else "N/A"}')
print(f'Pre-learning accuracy:  {pre_acc:.1%}')
print(f'Post-learning accuracy: {post_acc:.1%}')
print(f'Learning gain:          {gain:+.1%}')
print(f'Learning efficiency:    {eff:.1%}')
print()

# Per difficulty label
print('By Difficulty:')
print('-' * 60)
for label in sorted(results['difficulty_label'].unique()):
    sub = results[results['difficulty_label'] == label]
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {label:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per complexity
print()
print('By Complexity:')
print('-' * 60)
for comp in ['simple', 'medium', 'complex']:
    sub = results[results['complexity'] == comp]
    if len(sub) == 0: continue
    b = sub['pre_correct'].mean()
    l = sub['post_correct'].mean()
    g = l - b
    print(f'  {comp:20s}  pre={b:.1%}  post={l:.1%}  gain={g:+.1%}  (n={len(sub)})')

# Per item
print()
print('Per-item detail:')
print('-' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    delta = '  HELPED' if not row['pre_correct'] and row['post_correct'] else \
            '  HURT' if row['pre_correct'] and not row['post_correct'] else ''
    print(f'  [{row["difficulty_label"]:18s}] {row["procedure_name"]:15s} '
          f'expected={row["expected"]:>6s}  pre="{row["pre_extracted"]}"({b})  '
          f'post="{row["post_extracted"]}"({l}){delta}')

print()
print(f'Timing: pre={results["pre_time_s"].mean():.1f}s  study={results["study_time_s"].mean():.1f}s  post={results["post_time_s"].mean():.1f}s')

In [ ]:
print('STUDY NOTES INSPECTION')
print('=' * 60)
for _, row in results.iterrows():
    b = 'Y' if row['pre_correct'] else 'N'
    l = 'Y' if row['post_correct'] else 'N'
    print(f'\n--- [{row["difficulty_label"]}] seed={row["seed"]} ---')
    print(f'Procedure: {row["procedure_name"]} | Item: {row["item_desc"]}')
    print(f'Expected: {row["expected"]}')
    print(f'Pre: "{row["pre_extracted"]}" ({b})  Post: "{row["post_extracted"]}" ({l})')
    print(f'Notes (first 300 chars): {row["study_notes"][:300]}')
    print('...' if len(str(row['study_notes'])) > 300 else '')

In [ ]:
!pip install -q matplotlib
import matplotlib.pyplot as plt

labels = sorted(results['difficulty_label'].unique())
pre_scores = [results[results['difficulty_label']==l]['pre_correct'].mean() for l in labels]
post_scores = [results[results['difficulty_label']==l]['post_correct'].mean() for l in labels]

x = range(len(labels))
width = 0.35
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax1 = axes[0]
ax1.bar([i-width/2 for i in x], pre_scores, width, label='Pre-learning', color='#4C72B0')
ax1.bar([i+width/2 for i in x], post_scores, width, label='Post-learning', color='#55A868')
ax1.set_ylabel('Accuracy')
ax1.set_title('Trace-Based Imitation: Pre vs Post-Learning')
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2 = axes[1]
gains = [p-b for b,p in zip(pre_scores, post_scores)]
colors = ['#55A868' if g>=0 else '#C44E52' for g in gains]
ax2.bar(range(len(labels)), gains, color=colors)
ax2.set_ylabel('Learning Gain')
ax2.set_title('Learning Gain by Difficulty')
ax2.set_xticks(list(x))
ax2.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)

plt.tight_layout()
plt.savefig('trace_imitation_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Post-run: upload to BigQuery + Google Sheets via REST API ===
# Safe to do after benchmark — SDK can break, doesn't matter.

results_upload = pd.read_sql('SELECT * FROM runs', db)
db.close()
print(f'SQLite closed. {len(results_upload)} rows to upload.\n')

auth_token = None
gcp_project = None
try:
    import google.auth
    import google.auth.transport.requests
    creds, project = google.auth.default()
    creds.refresh(google.auth.transport.requests.Request())
    auth_token = creds.token
    gcp_project = project
    print(f'Authenticated. Project: {gcp_project}')
except Exception as e:
    print(f'Google auth not available: {e}')

BQ_DATASET = 'benchmark_runs'

# --- BigQuery ---
if auth_token and gcp_project:
    BQ_TABLE = DB_PATH.replace('_runs.db', '')
    try:
        import urllib.parse
        ds_url = f'https://api.kaggle.com/api/v1/bigquery/projects/{gcp_project}/datasets'
        ds_body = json.dumps({'datasetReference': {'datasetId': BQ_DATASET}, 'location': 'US'}).encode()
        req = urllib.request.Request(ds_url, data=ds_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        schema_fields = [{'name': c, 'type': 'STRING' if results_upload[c].dtype == 'object' else
                          'INTEGER' if 'correct' in c or c in ('seed','id','n_traces') else 'FLOAT'}
                         for c in results_upload.columns]
        create_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables'
        create_body = json.dumps({'tableReference': {'projectId': gcp_project, 'datasetId': BQ_DATASET, 'tableId': BQ_TABLE},
                                  'schema': {'fields': schema_fields}}).encode()
        req = urllib.request.Request(create_url, data=create_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        try: urllib.request.urlopen(req)
        except urllib.error.HTTPError as e:
            if e.code != 409: raise

        table_url = f'https://bigquery.googleapis.com/bigquery/v2/projects/{gcp_project}/datasets/{BQ_DATASET}/tables/{BQ_TABLE}/insertAll'
        rows_json = [{'json': {c: str(v) if pd.notna(v) else '' for c, v in row.items()}} for _, row in results_upload.iterrows()]
        insert_body = json.dumps({'rows': rows_json}).encode()
        req = urllib.request.Request(table_url, data=insert_body, method='POST',
                                     headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'BigQuery: {len(results_upload)} rows -> {gcp_project}.{BQ_DATASET}.{BQ_TABLE}')
    except Exception as e:
        print(f'BigQuery failed (non-fatal): {e}')

# --- Google Sheets ---
if auth_token:
    SHEET_NAME = f'Learning Bench — {BQ_TABLE} Runs'
    try:
        import urllib.parse
        search_url = ('https://www.googleapis.com/drive/v3/files'
                      f'?q=name%3D%27{urllib.parse.quote(SHEET_NAME)}%27+and+mimeType%3D%27application/vnd.google-apps.spreadsheet%27'
                      '&fields=files(id,webViewLink)')
        req = urllib.request.Request(search_url, headers={'Authorization': f'Bearer {auth_token}'})
        resp = json.loads(urllib.request.urlopen(req).read())
        files = resp.get('files', [])
        if files:
            sid = files[0]['id']
        else:
            create_url = 'https://sheets.googleapis.com/v4/spreadsheets'
            req = urllib.request.Request(create_url, data=json.dumps({'properties': {'title': SHEET_NAME}}).encode(),
                                         method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            resp = json.loads(urllib.request.urlopen(req).read())
            sid = resp['spreadsheetId']
            header = list(results_upload.columns)
            body = json.dumps({'values': [header]}).encode()
            req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                         data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
            urllib.request.urlopen(req)

        data_rows = [[str(v) if pd.notna(v) else '' for v in row] for _, row in results_upload.iterrows()]
        body = json.dumps({'values': data_rows}).encode()
        req = urllib.request.Request(f'https://sheets.googleapis.com/v4/spreadsheets/{sid}/values/Sheet1!A1:append?valueInputOption=RAW',
                                     data=body, method='POST', headers={'Authorization': f'Bearer {auth_token}', 'Content-Type': 'application/json'})
        urllib.request.urlopen(req)
        print(f'Sheets: {len(results_upload)} rows -> "{SHEET_NAME}"')
    except Exception as e:
        print(f'Sheets failed (non-fatal): {e}')

print(f'\nDone. SQLite: {DB_PATH}')